In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import IPython.display as ipd
from scipy import signal


def gen_noise(Ln,cutoff):    
    Tdur = 3 # noise duration (seconds)
    fs = 44100 # sample rate (Hz)
    dt = 1/fs  # time step
    tx = np.arange(0,Tdur,dt)  # time axis
    nsamp = len(tx)
    amp = 10**(Ln/20) # amplitude of the noise
    noise = amp*np.random.normal(0,1,nsamp) # broad band noise

    # make a low pass filter up to 10 kHz to make the noise less annoying 
    
    normalCutoff = cutoff / (fs/2)
    order = 15
    bLP, aLP = signal.butter(order, normalCutoff, btype='low')

    fnoise = signal.lfilter(bLP, aLP, noise)

    return fnoise, tx
#ipd.display(ipd.Audio(hpitch, rate=fs))


#fig, ax = plt.subplots(figsize=(13, 6))
#ax.set_xlabel('time')
#ax.set_ylabel('amplitude')
     
#ax.plot(tx, amp*noise, lw=3, c='r')
#tone = gen_tone(amp,1e3,tx,0)
#ax.plot(tx, tone, lw=3, c='r')

def gen_tone(amp,freq,tx,phase):
    '''generate tone '''
    
    tone = amp*np.sin(2*np.pi*freq*tx + phase)
    # make a ramp (fade in fade out)
    Rdur = 50e-3  # ramp duration
    x = np.arange(0,Rdur,tx[1]-tx[0])
    x = np.pi*x/Rdur
    rampUp = (1 + np.cos(x + np.pi))/2; # raised cosine onset
    rampDown = np.flip(rampUp)
    
    wholeramp = np.concatenate((rampUp, np.ones(len(tone)-2*len(x)), rampDown))
    
    tone = wholeramp*tone

        
    return tone

def update_signal(Lt):
    global fnoise, tx
    ampt = 10**(Lt/20)
    freq = 1e3
    tone_l = gen_tone(ampt,freq,tx,0)
    tone_r = gen_tone(ampt,freq,tx,0)
    signal_l = tone_l + fnoise
    signal_r = tone_r + fnoise
    signal = [signal_l, signal_r]  # pitch sensation
    
    #fig, ax = plt.subplots(figsize=(13, 6))
    #ax.set_xlabel('time (seconds)')
    #ax.set_ylabel('Amplitude')
     
    #ax.plot(tx,signal, lw=3, c='r')
    fs = 44100
    display(ipd.Audio(signal, rate=fs, autoplay=True))
    #ipd.display(ipd.Audio(signal, rate=fs,normalize=False))

global fnoise, tx

fnoise,tx = gen_noise(-20,2e3)


#c_slide = widgets.IntSlider(min=0,max=180,step=10,description='phase diff')
s_slide = widgets.IntSlider(min=-60,max=-30,step=1,description='tone level')

widgets.interact(update_signal, Lt=s_slide)








interactive(children=(IntSlider(value=-30, description='tone level', max=-30, min=-60), Output()), _dom_classe…

<function __main__.update_signal(Lt)>